# **FACT ORDERS**


### **Data Reading**

In [0]:
df_orders = spark.sql("select * from databricks_ete_project.silver.silver_orders")

In [0]:
df_dimcus = spark.sql("select dim_customer_key, customer_id as dim_customer_id from databricks_ete_project.gold.gold_customers")

df_dimprod = spark.sql("select product_id as dim_product_key, product_id as dim_product_id from databricks_ete_project.gold.gold_products")

### **Fact Dataframe**

In [0]:
df_fact = df_orders.join(df_dimcus, df_orders.customer_id == df_dimcus.dim_customer_id, 'left').join(df_dimprod, df_orders.product_id == df_dimprod.dim_product_id, 'left')

df_fact_new = df_fact.drop('dim_customer_id', 'dim_product_id', 'customer_id', 'product_id')



In [0]:
df_fact_new = df_fact_new.drop('_rescued_data')

In [0]:
df_fact_new.display()


### **Upsert on Fact Table**

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('databricks_ete_project.gold.gold_orders'):
    
    dlt_obj = DeltaTable.forName(spark, "databricks_ete_project.gold.gold_orders")
    
    dlt_obj.alias("trg").merge(
        df_fact_new.alias("src"),
        "trg.order_id = src.order_id AND trg.dim_customer_key = src.dim_customer_key AND trg.dim_product_key = src.dim_product_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
    
else:
    df_fact_new.write.format('delta')\
                     .mode("overwrite")\
                     .option("overwriteSchema", "true")\
                     .saveAsTable("databricks_ete_project.gold.gold_orders")


In [0]:
%sql
select * from databricks_ete_project.gold.gold_orders